# Análisis exploratorio de `tracks_base`

En esta sección se analiza la versión enriquecida del dataset, `tracks_base`.

El objetivo es estudiar la distribución de la popularidad musical y comenzar a identificar patrones asociados con canciones populares. Para ello se analizarán:

- la distribución general de `popularity`;
- la cantidad de canciones clasificadas como hits y no hits;
- la distribución de canciones por clase de popularidad;
- tendencias temporales por año y década;
- diferencias promedio en variables acústicas entre canciones hit y no hit.

Este análisis es importante porque antes de construir un `Hit Score` o un sistema de búsqueda de canciones similares, primero necesitamos entender cómo se comportan las variables musicales dentro del dataset.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("HitMakerAI_EDA")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 23:15:40 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/03 23:15:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 23:15:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
tracks_base = spark.read.parquet("../processed/tracks_base.parquet")

tracks_base.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- is_hit: integer (nullable = true)
 |--

In [5]:
tracks_base.select(
    F.count("*").alias("total_filas"),
    F.countDistinct("id").alias("ids_unicos")
).show()

[Stage 1:>                                                          (0 + 8) / 8]

+-----------+----------+
|total_filas|ids_unicos|
+-----------+----------+
|     586264|    586264|
+-----------+----------+



### Estadisticas generales

In [7]:
tracks_base.select(
    F.min("popularity").alias("min_popularity"),
    F.max("popularity").alias("max_popularity"),
    F.round(F.avg("popularity"), 2).alias("avg_popularity"),
    F.expr("percentile_approx(popularity, 0.5)").alias("median_popularity")
).show()

+--------------+--------------+--------------+-----------------+
|min_popularity|max_popularity|avg_popularity|median_popularity|
+--------------+--------------+--------------+-----------------+
|             0|           100|         27.57|               27|
+--------------+--------------+--------------+-----------------+



In [6]:
total_canciones = tracks_base.count()

popularity_class_dist = (
    tracks_base
    .groupBy("popularity_class")
    .agg(F.count("*").alias("num_canciones"))
    .withColumn(
        "porcentaje",
        F.round(F.col("num_canciones") / F.lit(total_canciones) * 100, 2)
    )
    .orderBy("popularity_class")
)

popularity_class_dist.show(truncate=False)

+----------------+-------------+----------+
|popularity_class|num_canciones|porcentaje|
+----------------+-------------+----------+
|alta            |7314         |1.25      |
|baja            |428759       |73.13     |
|media           |150191       |25.62     |
+----------------+-------------+----------+



### Popularidad promedio y proporción de hits por década

En esta sección se analiza la popularidad musical agrupando las canciones por década de lanzamiento.

Para cada década se calculan tres métricas principales:

- `num_canciones`: número de canciones disponibles en el dataset para esa década.
- `avg_popularity`: popularidad promedio de las canciones.
- `hit_rate`: proporción de canciones clasificadas como hit, donde `is_hit = 1` si `popularity >= 70`.

Los resultados muestran que la popularidad promedio tiende a ser mayor en décadas recientes. También se observa un aumento en la proporción de canciones clasificadas como hit en las décadas más recientes, especialmente en 2010 y 2020.

Sin embargo, la década de 2020 debe interpretarse con cuidado, ya que el dataset fue publicado alrededor de 2021 y por lo tanto solo contiene los primeros años de esa década. Esto significa que la década de 2020 no es comparable directamente con décadas completas como 1990, 2000 o 2010.

Además, la variable `popularity` de Spotify refleja la popularidad dentro de la plataforma y puede estar influenciada por escuchas recientes. Por esta razón, las canciones más nuevas pueden tener una ventaja frente a canciones antiguas.

Por lo tanto, este análisis no debe interpretarse como una comparación histórica definitiva entre décadas, sino como una exploración de cómo se distribuye la popularidad dentro del dataset disponible.

In [8]:
hit_dist = (
    tracks_base
    .groupBy("is_hit")
    .agg(F.count("*").alias("num_canciones"))
    .withColumn(
        "porcentaje",
        F.round(F.col("num_canciones") / F.lit(total_canciones) * 100, 2)
    )
    .orderBy("is_hit")
)

hit_dist.show(truncate=False)

+------+-------------+----------+
|is_hit|num_canciones|porcentaje|
+------+-------------+----------+
|0     |578950       |98.75     |
|1     |7314         |1.25      |
+------+-------------+----------+



La variable `is_hit` permite separar las canciones en dos grupos: canciones que cumplen con el umbral experimental de hit y canciones que no.

Dado que el umbral `popularity >= 70` corresponde aproximadamente al percentil 99 de popularidad, se espera que la clase de hits sea mucho más pequeña. Esto no representa un error, sino una decisión metodológica: el proyecto busca estudiar canciones excepcionalmente populares, no canciones promedio.

In [9]:
popularity_by_decade = (
    tracks_base
    .filter(F.col("release_decade").isNotNull())
    .groupBy("release_decade")
    .agg(
        F.count("*").alias("num_canciones"),
        F.round(F.avg("popularity"), 2).alias("avg_popularity"),
        F.round(F.avg("is_hit"), 4).alias("hit_rate")
    )
    .orderBy("release_decade")
)

popularity_by_decade.show(100, truncate=False)

+--------------+-------------+--------------+--------+
|release_decade|num_canciones|avg_popularity|hit_rate|
+--------------+-------------+--------------+--------+
|1900          |1            |19.0          |0.0     |
|1920          |7577         |1.14          |0.0     |
|1930          |13014        |2.11          |0.0     |
|1940          |17992        |1.78          |0.0     |
|1950          |35344        |8.39          |1.0E-4  |
|1960          |47235        |17.9          |0.0021  |
|1970          |61824        |24.17         |0.004   |
|1980          |82304        |25.68         |0.0034  |
|1990          |108850       |29.46         |0.0034  |
|2000          |86824        |36.63         |0.0086  |
|2010          |105095       |39.16         |0.0319  |
|2020          |20204        |41.73         |0.1097  |
+--------------+-------------+--------------+--------+



Aquí `hit_rate` representa la proporción de canciones hit dentro de cada década
e.g. el **3.19% de las canciones de la decada de 2010 son hits**

In [11]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_min"
]

hit_audio_profile = (
    tracks_base
    .groupBy("is_hit")
    .agg(
        *[
            F.round(F.avg(feature), 4).alias(f"avg_{feature}")
            for feature in audio_features
        ],
        F.count("*").alias("num_canciones")
    )
    .orderBy("is_hit")
)
stack_expr = "stack({}, {}) as (feature, avg_value)".format(
    len(audio_features),
    ", ".join([
        f"'{feature}', avg_{feature}"
        for feature in audio_features
    ])
)

hit_audio_profile_long = (
    hit_audio_profile
    .select(
        "is_hit",
        F.expr(stack_expr)
    )
)

hit_audio_profile_columnar = (
    hit_audio_profile_long
    .groupBy("feature")
    .pivot("is_hit", [0, 1])
    .agg(F.first("avg_value"))
    .withColumnRenamed("0", "no_hit")
    .withColumnRenamed("1", "hit")
    .withColumn(
        "diferencia_hit_vs_no_hit",
        F.round(F.col("hit") - F.col("no_hit"), 4)
    )
)

hit_audio_profile_columnar.show(truncate=False)

+----------------+--------+--------+------------------------+
|feature         |no_hit  |hit     |diferencia_hit_vs_no_hit|
+----------------+--------+--------+------------------------+
|acousticness    |0.4523  |0.2456  |-0.2067                 |
|valence         |0.553   |0.5213  |-0.0317                 |
|danceability    |0.5628  |0.6509  |0.0881                  |
|tempo           |118.4901|122.0407|3.5506                  |
|energy          |0.541   |0.6437  |0.1027                  |
|liveness        |0.2144  |0.1744  |-0.04                   |
|duration_min    |3.8374  |3.5953  |-0.2421                 |
|instrumentalness|0.1144  |0.0229  |-0.0915                 |
|speechiness     |0.105   |0.1006  |-0.0044                 |
|loudness        |-10.2433|-6.6776 |3.5657                  |
+----------------+--------+--------+------------------------+



Los resultados muestran diferencias interesantes entre ambos grupos. En promedio, las canciones hit presentan mayor `danceability`, mayor `energy`, mayor `tempo` y mayor `loudness`. Esto sugiere que las canciones más populares dentro del dataset tienden a ser más bailables, más energéticas y con mayor intensidad sonora.

Por otro lado, las canciones hit presentan menor `acousticness` e `instrumentalness`. Esto indica que, dentro del dataset analizado, los hits tienden a ser menos acústicos y menos instrumentales que el resto de las canciones.

También se observa que las canciones hit tienen una duración promedio ligeramente menor. Esto puede ser relevante para el análisis musical, ya que muchas canciones populares suelen tener estructuras más compactas y directas.

Es importante aclarar que estas diferencias son descriptivas y no implican causalidad. Es decir, no se puede concluir que una canción será hit únicamente por tener mayor energía o menor acousticness. Sin embargo, estos patrones son útiles para construir una primera aproximación al `Hit Score` experimental y para seleccionar variables relevantes en etapas posteriores del proyecto.

**Con columna de diferencia absoluta**

In [12]:
hit_audio_profile_columnar_ordered = (
    hit_audio_profile_columnar
    .withColumn(
        "diferencia_abs",
        F.round(F.abs(F.col("diferencia_hit_vs_no_hit")), 4)
    )
    .orderBy(F.desc("diferencia_abs"))
)

hit_audio_profile_columnar_ordered.show(truncate=False)

+----------------+--------+--------+------------------------+--------------+
|feature         |no_hit  |hit     |diferencia_hit_vs_no_hit|diferencia_abs|
+----------------+--------+--------+------------------------+--------------+
|loudness        |-10.2433|-6.6776 |3.5657                  |3.5657        |
|tempo           |118.4901|122.0407|3.5506                  |3.5506        |
|duration_min    |3.8374  |3.5953  |-0.2421                 |0.2421        |
|acousticness    |0.4523  |0.2456  |-0.2067                 |0.2067        |
|energy          |0.541   |0.6437  |0.1027                  |0.1027        |
|instrumentalness|0.1144  |0.0229  |-0.0915                 |0.0915        |
|danceability    |0.5628  |0.6509  |0.0881                  |0.0881        |
|liveness        |0.2144  |0.1744  |-0.04                   |0.04          |
|valence         |0.553   |0.5213  |-0.0317                 |0.0317        |
|speechiness     |0.105   |0.1006  |-0.0044                 |0.0044        |

### Comparación de características por clase de popularidad

In [15]:
class_audio_profile = (
    tracks_base
    .groupBy("popularity_class")
    .agg(
        *[
            F.round(F.avg(feature), 4).alias(f"avg_{feature}")
            for feature in audio_features
        ],
        F.count("*").alias("num_canciones")
    )
    .orderBy("popularity_class")
)


In [18]:
from itertools import chain
from pyspark.sql import functions as F

# Convertir de formato ancho a formato largo
stack_expr = "stack({}, {}) as (feature, avg_value)".format(
    len(audio_features),
    ", ".join([
        f"'{feature}', avg_{feature}"
        for feature in audio_features
    ])
)

class_audio_profile_long = (
    class_audio_profile
    .select(
        "popularity_class",
        F.expr(stack_expr)
    )
)

# Convertir a formato columnar: una fila por variable, columnas por clase
class_audio_profile_columnar = (
    class_audio_profile_long
    .groupBy("feature")
    .pivot("popularity_class", ["baja", "media", "alta"])
    .agg(F.first("avg_value"))
    .withColumn(
        "dif_alta_vs_baja",
        F.round(F.col("alta") - F.col("baja"), 4)
    )
    .withColumn(
        "dif_alta_vs_media",
        F.round(F.col("alta") - F.col("media"), 4)
    )
)

# Mantener el orden original de audio_features
order_mapping = F.create_map(
    list(chain.from_iterable([
        (F.lit(feature), F.lit(i))
        for i, feature in enumerate(audio_features)
    ]))
)

class_audio_profile_columnar = (
    class_audio_profile_columnar
    .withColumn("orden", order_mapping[F.col("feature")])
    .orderBy("orden")
    .drop("orden")
)

class_audio_profile_columnar.show(truncate=False)

+----------------+--------+--------+--------+----------------+-----------------+
|feature         |baja    |media   |alta    |dif_alta_vs_baja|dif_alta_vs_media|
+----------------+--------+--------+--------+----------------+-----------------+
|danceability    |0.5494  |0.6012  |0.6509  |0.1015          |0.0497           |
|energy          |0.5106  |0.6278  |0.6437  |0.1331          |0.0159           |
|loudness        |-11.0028|-8.0752 |-6.6776 |4.3252          |1.3976           |
|speechiness     |0.1124  |0.0839  |0.1006  |-0.0118         |0.0167           |
|acousticness    |0.5019  |0.3107  |0.2456  |-0.2563         |-0.0651          |
|instrumentalness|0.1335  |0.06    |0.0229  |-0.1106         |-0.0371          |
|liveness        |0.2202  |0.1977  |0.1744  |-0.0458         |-0.0233          |
|valence         |0.5559  |0.5447  |0.5213  |-0.0346         |-0.0234          |
|tempo           |117.5855|121.0724|122.0407|4.4552          |0.9683           |
|duration_min    |3.8037  |3

En esta tabla se comparan las características acústicas promedio de las canciones clasificadas como popularidad baja, media y alta.

El formato columnar permite observar con mayor claridad cómo cambia cada variable conforme aumenta la popularidad. Las columnas `dif_alta_vs_baja` y `dif_alta_vs_media` muestran la diferencia promedio entre canciones de popularidad alta y los otros grupos.

Los resultados muestran que las canciones de popularidad alta tienden a presentar mayor `danceability`, mayor `energy`, mayor `loudness` y mayor `tempo`. Esto sugiere que, dentro del dataset, las canciones más populares suelen ser más bailables, más energéticas y con mayor intensidad sonora.

Por otro lado, las canciones de popularidad alta presentan menor `acousticness`, menor `instrumentalness` y menor `liveness`. Esto indica que las canciones más populares tienden a ser menos acústicas, menos instrumentales y menos orientadas a grabaciones en vivo.

También se observa que la duración promedio disminuye en la clase de popularidad alta frente a la clase media y baja, lo cual puede sugerir que las canciones más populares tienden a tener una estructura más compacta.

Estas diferencias son descriptivas y no implican causalidad. Sin embargo, son útiles para identificar variables candidatas para etapas posteriores del proyecto, como la construcción de un `Hit Score` experimental o un sistema de búsqueda de canciones similares a hits.

Es importante considerar que las variables están en escalas distintas: algunas están entre 0 y 1, `tempo` está en BPM y `loudness` está en decibeles. Por esta razón, antes de construir una métrica compuesta será necesario normalizar las variables seleccionadas.

## Relación entre características acústicas y popularidad

Después de comparar los promedios de las características acústicas por clase de popularidad, el siguiente paso es analizar qué variables tienen mayor relación lineal con la variable `popularity`.

Para esto se calcula la correlación entre `popularity` y cada característica musical seleccionada. La correlación permite identificar si una variable tiende a aumentar o disminuir cuando aumenta la popularidad.

Es importante aclarar que la correlación no implica causalidad. Una correlación positiva no significa que una característica cause mayor popularidad, sino que ambas variables tienden a moverse en la misma dirección dentro del dataset.

Este análisis es útil para seleccionar variables candidatas para la construcción posterior de un `Hit Score` experimental.

In [20]:
from pyspark.sql import functions as F

audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_min"
]

correlation_profile = tracks_base.select(
    *[
        F.round(F.corr("popularity", feature), 4).alias(feature)
        for feature in audio_features
    ]
)


In [21]:
stack_expr = "stack({}, {}) as (feature, correlation_with_popularity)".format(
    len(audio_features),
    ", ".join([
        f"'{feature}', {feature}"
        for feature in audio_features
    ])
)

correlation_profile_columnar = (
    correlation_profile
    .select(F.expr(stack_expr))
    .withColumn(
        "abs_correlation",
        F.round(F.abs(F.col("correlation_with_popularity")), 4)
    )
    .orderBy(F.desc("abs_correlation"))
)

correlation_profile_columnar.show(truncate=False)

26/05/04 00:05:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------+---------------------------+---------------+
|feature         |correlation_with_popularity|abs_correlation|
+----------------+---------------------------+---------------+
|acousticness    |-0.371                     |0.371          |
|loudness        |0.3285                     |0.3285         |
|energy          |0.3029                     |0.3029         |
|instrumentalness|-0.2372                    |0.2372         |
|danceability    |0.1878                     |0.1878         |
|tempo           |0.0718                     |0.0718         |
|liveness        |-0.0489                    |0.0489         |
|speechiness     |-0.0474                    |0.0474         |
|duration_min    |0.0275                     |0.0275         |
|valence         |0.0047                     |0.0047         |
+----------------+---------------------------+---------------+



**Correlación entre características acústicas y popularidad**

Para complementar la comparación de promedios por clase de popularidad, se calculó la correlación entre `popularity` y cada una de las características acústicas seleccionadas.

Los resultados muestran que las variables con mayor relación lineal con la popularidad son `acousticness`, `loudness`, `energy`, `instrumentalness` y `danceability`.

La variable `acousticness` presenta una correlación negativa con la popularidad, lo que indica que las canciones más acústicas tienden a ser menos populares dentro del dataset. De manera similar, `instrumentalness` también presenta una relación negativa, lo que sugiere que las canciones más instrumentales tienden a aparecer con menor popularidad.

Por el contrario, `loudness`, `energy` y `danceability` presentan correlaciones positivas. Esto indica que las canciones más populares tienden a ser, en promedio, más intensas sonoramente, más energéticas y más bailables.

Algunas variables como `tempo`, `liveness`, `speechiness`, `duration_min` y `valence` presentan correlaciones bajas, por lo que su relación lineal con la popularidad parece ser más débil.

Es importante aclarar que la correlación no implica causalidad. Estos resultados no prueban que una característica cause mayor popularidad, pero sí ayudan a identificar variables candidatas para construir un `Hit Score` experimental.


In [22]:
# Guardar tablas agregadas del EDA en formato Parquet

popularity_class_dist.write.mode("overwrite").parquet(
    "../processed/eda/popularity_class_dist.parquet"
)

hit_dist.write.mode("overwrite").parquet(
    "../processed/eda/hit_dist.parquet"
)

popularity_by_decade.write.mode("overwrite").parquet(
    "../processed/eda/popularity_by_decade.parquet"
)

hit_audio_profile_columnar.write.mode("overwrite").parquet(
    "../processed/eda/hit_audio_profile_columnar.parquet"
)

class_audio_profile_columnar.write.mode("overwrite").parquet(
    "../processed/eda/class_audio_profile_columnar.parquet"
)

correlation_profile_columnar.write.mode("overwrite").parquet(
    "../processed/eda/correlation_profile_columnar.parquet"
)

In [23]:
#En .csv para visualización
popularity_class_dist.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "../processed/eda_csv/popularity_class_dist"
)

hit_dist.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "../processed/eda_csv/hit_dist"
)

popularity_by_decade.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "../processed/eda_csv/popularity_by_decade"
)

hit_audio_profile_columnar.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "../processed/eda_csv/hit_audio_profile_columnar"
)

class_audio_profile_columnar.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "../processed/eda_csv/class_audio_profile_columnar"
)

correlation_profile_columnar.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "../processed/eda_csv/correlation_profile_columnar"
)